## 1. Instalacja i konfiguracja

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install librosa==0.11.0
%pip install mirdata==0.3.9
%pip install matplotlib==3.9.4
%pip install numpy==1.26.3
%pip install scikit-learn==1.6.1
%pip install tqdm==4.67.1
%pip install pretty_midi>=0.2.10
%pip install madmom==0.18.0
%pip install pypianoroll==1.0.2

In [42]:
# Podstawowe biblioteki
import os
import numpy as np
from tqdm import tqdm
import random
from datetime import datetime
from pprint import pprint

# Przetwarzanie dźwięku
import librosa  # Analiza audio
import mirdata  # Dataset GuitarSet

# Uczenie maszynowe
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch import optim

# Wizualizacja
import matplotlib.pyplot as plt

# Konfiguracja GPU - automatyczne wykrywanie
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")

Używane urządzenie: cuda
Model karty GPU: NVIDIA GeForce RTX 3070


## 2. Ładowanie i przygotowanie danych 

### 2.2. Pobranie i wczytanie guitarset

In [20]:
def initialize_guitarset(data_dir="guitarset_data"):
    """Initialize and download GuitarSet if needed"""
    guitarset = mirdata.initialize("guitarset", data_home=data_dir)
    
    # Check if data exists
    example_track = guitarset.track(guitarset.track_ids[0])
    if not os.path.exists(example_track.audio_mic_path):
        print("Downloading GuitarSet data...")
        guitarset.download()
    else:
        print("GuitarSet data already exists")
    
    return guitarset

guitarset = initialize_guitarset()


GuitarSet data already exists


### Midi z JAMS

In [41]:
import os
import pretty_midi
import jams

def simple_jams_to_midi(jams_path, output_midi_path):
    jam = jams.load(jams_path)
    midi = pretty_midi.PrettyMIDI()
    instrument_program = pretty_midi.instrument_name_to_program("Acoustic Guitar (nylon)")
    instrument = pretty_midi.Instrument(program=instrument_program, name='Classical Guitar')
    
    for annotation in jam.search(namespace='note_midi'):
        for note in annotation:
            pitch = int(note.value)
            start = float(note.time)
            end = float(note.time + note.duration)
            velocity = 100

            midi_note = pretty_midi.Note(
                velocity=velocity,
                pitch=pitch,
                start=start,
                end=end
            )
            instrument.notes.append(midi_note)
    
    midi.instruments.append(instrument)
    midi.write(output_midi_path)
    print(f"Zapisano plik MIDI: {output_midi_path}")


example_track = guitarset.track(guitarset.track_ids[2])
jams_path = example_track.jams_path
output_midi_path = "przyklad_midi.mid"

simple_jams_to_midi(jams_path, output_midi_path)

Zapisano plik MIDI: przyklad_midi.mid


### 2.3. Ekstrakcja danych treningowych

In [26]:
ANNOTATION_HOP = 0.01
FREQ_BINS = librosa.cqt_frequencies(n_bins=252, fmin=librosa.note_to_hz("A0"), bins_per_octave=36)

def augment_audio(audio, sr, pitch_shift_semitones=(-2, 2), time_stretch_factors=(0.95, 1.05), apply_prob=0.5):
    original_length = len(audio)

    # Pitch shift
    if random.random() < apply_prob:
        n_steps = random.uniform(*pitch_shift_semitones)
        audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

    # Time stretch
    if random.random() < apply_prob:
        rate = random.uniform(*time_stretch_factors)
        audio = librosa.effects.time_stretch(audio, rate=rate)

        # Przywróć do oryginalnej długości, by matryce adnotacji pasowały
        if len(audio) > original_length:
            audio = audio[:original_length]
        else:
            audio = np.pad(audio, (0, original_length - len(audio)))

    return audio

def time_scale(audio_length_sec):
    return np.arange(0, audio_length_sec, ANNOTATION_HOP)

def to_sparse_matrix(note_data, time_vector, freq_bins, onset_only=False):
    n_time = len(time_vector)
    n_freq = len(freq_bins)
    matrix = np.zeros((n_freq, n_time), dtype=np.float32)

    if onset_only:
        if hasattr(note_data, "intervals") and hasattr(note_data, "pitches"):
            times = note_data.intervals[:, 0]
            freqs = note_data.pitches
            for onset, freq in zip(times, freqs):
                t_idx = np.argmin(np.abs(time_vector - onset))
                f_idx = np.argmin(np.abs(freq_bins - freq))
                matrix[f_idx, t_idx] = 1.0
        else:
            raise ValueError("Brak obsługi onset_only dla tego typu danych.")
    elif hasattr(note_data, "intervals") and hasattr(note_data, "pitches"):
        for (start, end), freq in zip(note_data.intervals, note_data.pitches):
            t_idxs = np.where((time_vector >= start) & (time_vector <= end))[0]
            f_idx = np.argmin(np.abs(freq_bins - freq))
            matrix[f_idx, t_idxs] = 1.0
    elif hasattr(note_data, "frequency_list") and hasattr(note_data, "times"):
        for t, freqs in zip(note_data.times, note_data.frequency_list):
            t_idx = np.argmin(np.abs(time_vector - t))
            for freq in freqs:
                f_idx = np.argmin(np.abs(freq_bins - freq))
                matrix[f_idx, t_idx] = 1.0
    else:
        raise ValueError("Nieznany format adnotacji.")

    return matrix


def prepare_dataset_v2(guitarset, num_tracks=10, sr=22050, seq_length=256, augment=False):
    X_waveform, Y_pitch, Y_onset, Y_contour = [], [], [], []
    track_ids = guitarset.track_ids[:num_tracks]

    for track_id in tqdm(track_ids, desc="Przetwarzanie utworów"):
        track = guitarset.track(track_id)

        # Audio
        audio, _ = librosa.load(track.audio_mic_path, sr=sr, mono=True)
        if augment:
            audio = augment_audio(audio, sr)

        duration_sec = len(audio) / sr
        t_vec = time_scale(duration_sec)
        n_frames = len(t_vec)

        pitch_matrix = to_sparse_matrix(track.notes_all, t_vec, FREQ_BINS)
        onset_matrix = to_sparse_matrix(track.notes_all, t_vec, FREQ_BINS, onset_only=True)
        contour_matrix = to_sparse_matrix(track.multif0, t_vec, FREQ_BINS)

        # Dopasuj długość waveformu do liczby ramek
        total_samples = int(n_frames * ANNOTATION_HOP * sr)
        audio = audio[:total_samples]

        hop_length = int(ANNOTATION_HOP * sr)
        for frame_start in range(0, n_frames - seq_length, seq_length):
            sample_start = frame_start * hop_length
            sample_end = sample_start + seq_length * hop_length

            x = audio[sample_start:sample_end]
            if len(x) != seq_length * hop_length:
                continue

            X_waveform.append(x)
            Y_pitch.append(pitch_matrix[:, frame_start:frame_start + seq_length])
            Y_onset.append(onset_matrix[:, frame_start:frame_start + seq_length])
            Y_contour.append(contour_matrix[:, frame_start:frame_start + seq_length])

    X_tensor = torch.tensor(np.array(X_waveform), dtype=torch.float32).unsqueeze(1)  # (B, 1, T)
    Y_pitch_tensor = torch.tensor(np.array(Y_pitch), dtype=torch.float32)
    Y_onset_tensor = torch.tensor(np.array(Y_onset), dtype=torch.float32)
    Y_contour_tensor = torch.tensor(np.array(Y_contour), dtype=torch.float32)

    return TensorDataset(X_tensor, Y_pitch_tensor, Y_onset_tensor, Y_contour_tensor)

dataset = prepare_dataset_v2(guitarset, num_tracks=360, sr=22050, seq_length=256, augment=True)
print(f"Przygotowano {len(dataset)} sekwencji.")


Przetwarzanie utworów: 100%|██████████| 360/360 [24:31<00:00,  4.09s/it]


Przygotowano 4092 sekwencji.


### 2.4 Wizualizacja

In [27]:
def visualize_example(cqt, pitch_matrix, onset_matrix, sr=44100, hop_length=256, fmin=librosa.note_to_hz('A0'), bins_per_octave=36):
    fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    # CQT
    img1 = librosa.display.specshow(cqt, sr=sr, hop_length=hop_length,
                                    x_axis='time', y_axis='cqt_note', bins_per_octave=bins_per_octave,
                                    fmin=fmin, ax=axs[0])
    axs[0].set_title('CQT (log power)')
    fig.colorbar(img1, ax=axs[0], format="%+2.f dB")

    # Pitch Matrix
    img2 = axs[1].imshow(pitch_matrix, aspect='auto', origin='lower', cmap='gray_r')
    axs[1].set_title("Pitch matrix (binary)")

    # Onset Matrix
    img3 = axs[2].imshow(onset_matrix, aspect='auto', origin='lower', cmap='gray_r')
    axs[2].set_title("Onset matrix (binary)")

    axs[2].set_xlabel("Frames")
    plt.tight_layout()
    plt.show()


## 3. Model

In [35]:
def waveform_to_cqt(waveform, sr=22050, n_bins=252, bins_per_octave=36, hop_length=512):
    cqt = librosa.cqt(
        y=waveform.numpy(),
        sr=sr,
        hop_length=hop_length,
        n_bins=n_bins,
        bins_per_octave=bins_per_octave,
        fmin=librosa.note_to_hz("C1")
    )
    cqt_db = librosa.amplitude_to_db(np.abs(cqt))
    return torch.tensor(cqt_db, dtype=torch.float32)

class HarmonicStacking(nn.Module):
    def __init__(self, harmonics=[0.5, 1, 2, 3]):
        super().__init__()
        self.harmonics = harmonics

    def forward(self, x):
        B, F, T = x.shape
        stacked = [x]
        for h in self.harmonics:
            idx = torch.round(torch.arange(F) * h).long()
            idx = torch.clamp(idx, max=F - 1)
            harmonic = x[:, idx, :]
            stacked.append(harmonic)
        return torch.cat(stacked, dim=1)  # (B, freq * len(harmonics), time)

class EnhancedMusicTranscriberV2(nn.Module):
    def __init__(self, sample_rate=22050, n_bins=252, bins_per_octave=36, hidden_size=256, num_layers=2):
        super().__init__()
        self.sample_rate = sample_rate
        self.n_bins = n_bins
        self.bins_per_octave = bins_per_octave
        self.hop_length = 512

        self.harmonic_stack = HarmonicStacking([0.5, 1, 2])

        cnn_channels = 64
        self.conv_pre_rnn = nn.Sequential(
            nn.BatchNorm2d(1),
            nn.Conv2d(1, cnn_channels, kernel_size=(5, 5), padding=2),
            nn.ReLU(),
            nn.Conv2d(cnn_channels, cnn_channels, kernel_size=(3, 3), padding=1),
            nn.ReLU(),
        )

        self.rnn = nn.LSTM(
            input_size=n_bins * len(self.harmonic_stack.harmonics),
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.pitch_head = nn.Sequential(nn.Linear(hidden_size * 2, n_bins), nn.Sigmoid())
        self.onset_head = nn.Sequential(nn.Linear(hidden_size * 2, n_bins), nn.Sigmoid())
        self.contour_head = nn.Sequential(nn.Linear(hidden_size * 2, n_bins), nn.Sigmoid())

    def forward(self, waveform):
        x = self.feature_extractor(waveform)  # (B, C, F, T)
        x = x.squeeze(1) if x.shape[1] == 1 else x  # (B, F, T) jeśli C=1
        x = x.transpose(1, 2)  # (B, T, F)
        
        # Tu input_size = F
        x, _ = self.rnn(x)
    
        pitch = self.pitch_head(x)
        onset = self.onset_head(x)
        contour = self.contour_head(x)
    
        return {
            "pitch": pitch,
            "onset": onset,
            "contour": contour
        }



## 4. Trenowanie modelu

In [36]:
def train_model(
    model,
    dataset,
    batch_size=16,
    learning_rate=1e-3,
    epochs=100,
    val_split=0.1,
    output_dir="checkpoints",
    patience=20,
    device=None
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Używane urządzenie: {device}")

    model = model.to(device)

    # Podział na train / val
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)

    # Optymalizator, scheduler i kryteria
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

    criterion = nn.BCELoss()

    # Ścieżki do checkpointów
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    ckpt_dir = os.path.join(output_dir, timestamp)
    os.makedirs(ckpt_dir, exist_ok=True)
    best_val_loss = float("inf")
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [train]"):
            x, y_pitch, y_onset, y_contour = [b.to(device) for b in batch]

            optimizer.zero_grad()
            outputs = model(x)

            loss_pitch = criterion(outputs["pitch"], y_pitch)
            loss_onset = criterion(outputs["onset"], y_onset)
            loss_contour = criterion(outputs["contour"], y_contour)

            loss = loss_pitch + loss_onset + loss_contour
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Walidacja
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} [val]"):
                x, y_pitch, y_onset, y_contour = [b.to(device) for b in batch]
                outputs = model(x)

                loss_pitch = criterion(outputs["pitch"], y_pitch)
                loss_onset = criterion(outputs["onset"], y_onset)
                loss_contour = criterion(outputs["contour"], y_contour)
                loss = loss_pitch + loss_onset + loss_contour

                val_loss += loss.item()

        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        print(f"Epoch {epoch + 1} — train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(ckpt_dir, "best_model.pt"))
            print("Zapisano nowy najlepszy model!")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("⏹️ Wczesne zatrzymanie — brak poprawy walidacji.")
                break

    print(f"Trening zakończony. Najlepszy model zapisany w: {ckpt_dir}/best_model.pt")

model = EnhancedMusicTranscriberV2()
train_model(model, dataset, batch_size=16, learning_rate=1e-3, epochs=10, output_dir="training_logs")

Używane urządzenie: cuda


Epoch 1/10 [train]:   0%|          | 0/231 [00:00<?, ?it/s]


AttributeError: 'EnhancedMusicTranscriberV2' object has no attribute 'feature_extractor'

### 4.1 Wizualizacja

In [ ]:
def visualize_sample(model, dataset, idx=0, device='cpu'):
    model.eval()
    model.to(device)

    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    x, y_pitch, y_onset, y_contour = next(iter(torch.utils.data.Subset(dataloader, [idx])))

    x = x.to(device)
    with torch.no_grad():
        output = model(x)

    def plot_prediction(pred, target, title):
        plt.figure(figsize=(10, 4))
        plt.imshow(pred.squeeze().cpu(), aspect='auto', origin='lower', cmap='hot', alpha=0.6)
        plt.imshow(target.squeeze().cpu(), aspect='auto', origin='lower', cmap='Blues', alpha=0.4)
        plt.title(title)
        plt.xlabel('Czas (ramki)')
        plt.ylabel('Pasma częstotliwości')
        plt.colorbar(label='Aktywacja')
        plt.tight_layout()
        plt.show()

    plot_prediction(output['pitch'][0], y_pitch[0], '🎵 Pitch: Predykcja vs Ground Truth')
    plot_prediction(output['onset'][0], y_onset[0], '💥 Onset: Predykcja vs Ground Truth')
    plot_prediction(output['contour'][0], y_contour[0], '🎶 Contour: Predykcja vs Ground Truth')

# Przygotuj dane
dataset = prepare_dataset_v2(guitarset, num_tracks=5, augment=False)

# Załaduj model
model = EnhancedMusicTranscriberV2()
model.load_state_dict(torch.load("model.pth", map_location='cpu'))

# Wizualizacja
visualize_sample(model, dataset, idx=0)

